# LAV-DF private, authenticated baseline workflow

**Run setup and diagnostics in order. Do not train until both the environment and dataset audits pass.** Review provider terms and confirm that Colab and your private storage are permitted. Never put tokens, videos, features, identity metadata, reports or weights in public Git.

The Colab control kernel may show Python 3.13: this is expected. Model loading, extraction, dataset processing, training and evaluation run in a separate Python 3.12.10 virtual environment at `/content/lavdf-cuda-env/bin/python`. Its Python installation is under `/content/lavdf-python312-bootstrap`. The next diagnostic prints both interpreters and runs a CUDA calculation in the project environment. Select a T4 GPU and use a fresh Linux x86-64 Colab session with sufficient free storage and internet access for the setup downloads.

If setup is interrupted, rerun the setup cell: verified files and the virtual environment are reused, and package installation resumes. If it reports an unrecognized existing repository or environment, preserve any needed files and start a fresh Colab runtime; the setup intentionally does not delete unknown files. If a download or installation fails, check the error and connection, then rerun the cell. Private checkpoints and Hugging Face authentication are only needed in later cells.


In [ ]:
GPU_DIAGNOSTIC_PASSED = MODEL_SMOKE_PASSED = DATASET_VERIFIED = False
import hashlib, os, re, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

# Colab can set Python-specific paths for its control kernel; do not pass them to Python 3.12.
PROJECT_ENV = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME')}

REPO_COMMIT = 'f079dc36c6973a6e0645f111866316aab924586b'  # paste the exact reviewed 40-character commit from the instructions
REPO = Path('/content/deepfake-detection-dtw')
BOOTSTRAP = Path('/content/lavdf-python312-bootstrap')
VENV = Path('/content/lavdf-cuda-env')
PYTHON = str(VENV / 'bin/python')
PYTHON_SHA256 = '3b7de567f4a8a1c3e5be2d51af3023dd3c1075cfacb6cd8efb511e0ef4b31fa8'
PYTHON_URL = 'https://github.com/astral-sh/python-build-standalone/releases/download/20250529/cpython-3.12.10%2B20250529-x86_64-unknown-linux-gnu-install_only.tar.gz'
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT), 'Set REPO_COMMIT to the reviewed commit SHA'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Pradeepbeleri/deepfake-detection-dtw.git', str(REPO)], check=True)
assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--untracked-files=all'],text=True).strip(), 'Repository has local changes; preserve them and use a clean checkout'
subprocess.run(['git','-C',str(REPO),'fetch','origin',REPO_COMMIT],check=True)
subprocess.run(['git','-C',str(REPO),'switch','--detach',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == REPO_COMMIT

# Refuse to reuse an unrelated or partially created installation. Never overwrite it.
marker = BOOTSTRAP / '.verified-sha256'
if not BOOTSTRAP.exists():
    with tempfile.TemporaryDirectory(prefix='lavdf-python-', dir=BOOTSTRAP.parent) as temp:
        archive = Path(temp) / 'python.tar.gz'
        urllib.request.urlretrieve(PYTHON_URL, archive)
        with archive.open('rb') as stream:
            actual = hashlib.file_digest(stream, 'sha256').hexdigest()
        if actual != PYTHON_SHA256:
            raise RuntimeError('Python archive SHA-256 mismatch; no archive was used')
        # Extract and mark in a temporary sibling: interruption cannot leave a
        # partially extracted installation at the final path.
        staged = Path(temp) / 'bootstrap'
        staged.mkdir(mode=0o700)
        with tarfile.open(archive, 'r:gz') as tar:
            tar.extractall(path=staged, filter='data')
        (staged / '.verified-sha256').write_text(PYTHON_SHA256)
        staged.rename(BOOTSTRAP)
assert marker.is_file() and marker.read_text().strip() == PYTHON_SHA256, 'Existing bootstrap is incomplete or unrecognized. Preserve your files and start a fresh Colab runtime.'
standalone = BOOTSTRAP / 'python/bin/python3.12'
assert subprocess.check_output([str(standalone),'-c','import sys; print(".".join(map(str,sys.version_info[:3])))'],text=True,env=PROJECT_ENV).strip() == '3.12.10'
venv_marker = VENV / '.lavdf-python-sha256'
if not VENV.exists():
    # Mark our directory before provisioning so an interrupted venv can be resumed.
    VENV.mkdir(mode=0o700)
    venv_marker.write_text(PYTHON_SHA256)
assert venv_marker.is_file() and venv_marker.read_text().strip() == PYTHON_SHA256, 'Existing virtual environment is unrecognized. Preserve your files and start a fresh Colab runtime.'
venv_complete = VENV / '.venv-complete'
if not venv_complete.is_file() or not Path(PYTHON).is_file():
    subprocess.run([str(standalone),'-m','venv',str(VENV)],env=PROJECT_ENV,check=True)
    venv_complete.write_text('ok')
print('Project interpreter:', PYTHON, flush=True)

# GPU wheels and CUDA libraries are confined to this virtual environment.
cuda_packages = ['nvidia-cuda-nvrtc-cu12','nvidia-cuda-runtime-cu12','nvidia-cuda-cupti-cu12','nvidia-cudnn-cu12','nvidia-cublas-cu12','nvidia-cufft-cu12','nvidia-curand-cu12','nvidia-cusolver-cu12','nvidia-cusparse-cu12','nvidia-nccl-cu12','nvidia-nvtx-cu12','nvidia-nvjitlink-cu12','triton']
subprocess.run([PYTHON,'-m','pip','install','--index-url','https://pypi.org/simple','-c',str(REPO/'requirements-colab-cu124.txt'),*cuda_packages],env=PROJECT_ENV,check=True)
subprocess.run([PYTHON,'-m','pip','install','--no-deps','--index-url','https://download.pytorch.org/whl/cu124','torch==2.5.1+cu124','torchvision==0.20.1+cu124'],env=PROJECT_ENV,check=True)
subprocess.run([PYTHON,'-m','pip','install','-c',str(REPO/'requirements-colab-cu124.txt'),'-e',str(REPO),'huggingface_hub==0.30.2'],env=PROJECT_ENV,check=True)
subprocess.run([PYTHON,'-m','pip','check'],env=PROJECT_ENV,check=True)
print('Setup completed. Run the next cell to check both interpreters and CUDA.', flush=True)


In [ ]:
GPU_DIAGNOSTIC_PASSED = False
import os, platform, subprocess, sys
from pathlib import Path

print('=== Colab control kernel (not used for project ML) ===', flush=True)
print('Python version:', platform.python_version(), flush=True)
print('Python executable:', sys.executable, flush=True)
print('Python 3.13 here is OK; the project uses the separate interpreter below.', flush=True)

PYTHON = '/content/lavdf-cuda-env/bin/python'
if not Path(PYTHON).is_file():
    raise RuntimeError('Project interpreter not found. Run the setup cell above and resolve any error before retrying.')

DIAGNOSTIC = r"""
import importlib.metadata as md, platform, sys
import torch, torchvision, whisper, mediapipe
import deepfake_dtw.encoders
print('\n=== Project ML environment (used by project commands) ===', flush=True)
print('Python version:', platform.python_version(), flush=True)
print('Python executable:', sys.executable, flush=True)
print('PyTorch:', torch.__version__, flush=True)
print('torchvision:', torchvision.__version__, flush=True)
print('PyTorch CUDA build:', torch.version.cuda, flush=True)
print('Whisper:', md.version('openai-whisper'), '| MediaPipe:', mediapipe.__version__, flush=True)
available = torch.cuda.is_available()
print('CUDA available:', available, flush=True)
print('GPU name:', torch.cuda.get_device_name(0) if available else 'Unavailable', flush=True)
if sys.version_info[:3] != (3, 12, 10):
    raise RuntimeError('Unexpected project Python version; rerun setup or use a fresh runtime.')
if torch.__version__ != '2.5.1+cu124' or torchvision.__version__ != '0.20.1+cu124' or torch.version.cuda != '12.4':
    raise RuntimeError('Unexpected PyTorch CUDA build; rerun setup and check pip errors.')
if not available:
    raise RuntimeError('CUDA unavailable in the project interpreter. Select a T4 GPU in Runtime > Change runtime type, reconnect, rerun setup and check the runtime driver.')
x = torch.tensor([1.0, 2.0], device='cuda')
if (x*x).sum().item() != 5.0:
    raise RuntimeError('CUDA tensor calculation returned an unexpected result.')
torch.cuda.synchronize()
print('CUDA tensor operation: PASSED', flush=True)
"""
try:
    subprocess.run([PYTHON, '-c', DIAGNOSTIC], env={k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME')}, check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Project diagnostic failed; see the error above. Resolve it and rerun this cell before continuing.') from exc
GPU_DIAGNOSTIC_PASSED = True
print('Environment check passed. Continue to private checkpoint configuration.', flush=True)


In [ ]:
import os, json, subprocess, sys
from pathlib import Path

PROJECT_ENV = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME')}

REPO = Path('/content/deepfake-detection-dtw')
PYTHON = '/content/lavdf-cuda-env/bin/python'
PRIVATE = Path('/content/drive/MyDrive/deepfake-private')  # only if provider/institution authorize this storage
MEDIA = PRIVATE / 'lavdf' / 'media'
FEATURES = PRIVATE / 'lavdf' / 'features'
REPORTS = PRIVATE / 'lavdf' / 'reports'
PROVENANCE = PRIVATE / 'lavdf' / 'curated-provenance.json'
PLAN = PRIVATE / 'lavdf' / 'plan.json'
METADATA_LOCAL = PRIVATE / 'lavdf' / 'metadata.min.json'
WHISPER = PRIVATE / 'tiny.en.pt'
DINO = PRIVATE / 'dinov2_vits14_pretrain.pth'
DINO_SOURCE = PRIVATE / 'dinov2-src'  # clean pinned official checkout
DINO_SHA256 = ''  # full hash of your officially obtained file, verified after transfer
assert (REPO / 'src/deepfake_dtw/lavdf.py').exists() and Path(PYTHON).is_file(), 'Complete repository environment setup first'
assert PRIVATE.is_dir(), 'Mount and create your authorized private storage root before continuing'
assert len(DINO_SHA256) == 64 and WHISPER.is_file() and DINO.is_file(), 'Provide and verify both permitted model files'
for directory in (MEDIA, FEATURES, REPORTS):
    directory.mkdir(parents=True, mode=0o700, exist_ok=True)


In [ ]:
MODEL_SMOKE_PASSED = False
assert globals().get('GPU_DIAGNOSTIC_PASSED', False), 'Run the GPU diagnostic successfully first'
# Required model-loading check. It performs no classifier training.
MODEL_SMOKE = "import json, numpy as np, sys, torch\nfrom deepfake_dtw.encoders import WhisperAudioEncoder, DinoVisualEncoder\nfrom deepfake_dtw.preprocessing import prepare_audio, prepare_video\ndevice=torch.device('cuda')\na=WhisperAudioEncoder(device=device,mode='pretrained',checkpoint=sys.argv[1])\nv=DinoVisualEncoder(device=device,checkpoint=sys.argv[2],sha256=sys.argv[3],source_dir=sys.argv[4])\naudio=prepare_audio(np.zeros(16000,dtype=np.float32),16000)\nvideo=prepare_video(np.zeros((1,64,64,3),dtype=np.uint8),[0.0])\naf=a.extract(audio); vf=v.extract(video)\nassert torch.isfinite(af.values).all() and torch.isfinite(vf.values).all()\nassert af.values.shape==(1,1500,384) and vf.values.shape==(1,384)\ntorch.cuda.synchronize()\nprint(json.dumps({'model_interpreter':sys.executable,'device':str(device),'gpu':torch.cuda.get_device_name(0),'audio_shape':list(af.values.shape),'video_shape':list(vf.values.shape),'audio_parameters':sum(p.numel() for p in a.model.parameters()),'visual_parameters':sum(p.numel() for p in v.model.parameters()),'gpu_peak_bytes':torch.cuda.max_memory_allocated()},indent=2))\n"
model_env = dict(PROJECT_ENV, XFORMERS_DISABLED='1', OMP_NUM_THREADS='2', CUBLAS_WORKSPACE_CONFIG=':4096:8')
subprocess.run([PYTHON,'-c',MODEL_SMOKE,str(WHISPER),str(DINO),DINO_SHA256,str(DINO_SOURCE)],env=model_env,check=True)
MODEL_SMOKE_PASSED = True


## Provider access (interactive)

Use the secure prompt below only after confirming your provider access and permitted storage. The token is passed to the isolated interpreter through standard input and cached only under the private ephemeral `HF_HOME`; it is not embedded in notebook cells, command arguments or Git. Clear that private cache after use. Do not download the full dataset by default.


In [ ]:
# Authenticate in the isolated interpreter; no packages are installed into the Colab kernel.
from getpass import getpass
HF_HOME = Path('/content/lavdf-hf-private')
HF_HOME.mkdir(mode=0o700, exist_ok=True)
os.chmod(HF_HOME, 0o700)
hf_env = dict(PROJECT_ENV, HF_HOME=str(HF_HOME))
token = getpass('Hugging Face read token (not echoed): ')
try:
    subprocess.run([PYTHON,'-c',"import sys; from huggingface_hub import login; login(token=sys.stdin.read().strip(), add_to_git_credential=False)"],input=token+'\n',text=True,env=hf_env,check=True)
finally:
    del token
FILES_JSON = HF_HOME / 'repo-files.json'
list_script = "import json,sys; from pathlib import Path; from huggingface_hub import HfApi; Path(sys.argv[1]).write_text(json.dumps(HfApi().list_repo_files('ControlNet/LAV-DF', repo_type='dataset')))"
subprocess.run([PYTHON,'-c',list_script,str(FILES_JSON)],env=hf_env,check=True)
files = json.loads(FILES_JSON.read_text())
print('Provider file count:',len(files))
print('Candidate metadata filenames:',[x for x in files if 'metadata' in x.lower()][:20])


## Obtain metadata and curate provenance

Choose the exact official metadata filename after inspecting the listing. If no JSON metadata is directly downloadable, obtain it using the official provider instructions. The sidecar is a private JSON dictionary keyed by exact provider `file` path. Each entry must contain `source_identity`, `target_identity` (required for manipulated examples), `source_recordings` (include provider `original` exactly for manipulated examples), and `usage_rights`. Identity and source links must come from audited provider evidence; do not infer or fabricate them. The official metadata itself does not establish all these fields.


In [ ]:
METADATA_FILENAME = ''  # exact provider filename, selected after listing
assert METADATA_FILENAME in files and METADATA_FILENAME.endswith('.json')
download_script = "import sys; from huggingface_hub import hf_hub_download; print(hf_hub_download(repo_id='ControlNet/LAV-DF', repo_type='dataset', filename=sys.argv[1], local_dir=sys.argv[2]))"
metadata_path = subprocess.check_output([PYTHON,'-c',download_script,METADATA_FILENAME,str(MEDIA)],env=hf_env,text=True).strip()
assert PROVENANCE.is_file(), 'Supply the independently audited private provenance sidecar before planning'
assert not PLAN.exists(), 'Choose a new plan path; no overwrites'
subprocess.run([PYTHON, '-m', 'deepfake_dtw.lavdf', '--metadata', metadata_path, '--provenance', str(PROVENANCE), '--media-root', str(MEDIA), '--feature-root', str(FEATURES), '--output', str(PLAN), '--limit-per-split', '60', '--seed', '42'], cwd=REPO, env=PROJECT_ENV, check=True)


## Download only the selected official media

The plan preserves official splits and whole identity/source groups. If listed filenames are not direct Hugging Face repository files, stop and use an authorized provider-supported archive workflow; do not download arbitrary or unlisted mirrors. Check available storage and identity overlaps before proceeding.


In [ ]:
records = json.loads(PLAN.read_text())
selected = sorted({r['provider_file'] for r in records})
missing = [name for name in selected if name not in files]
assert not missing, 'Selected videos are not direct provider files; use official authorized archive access instead'
for name in selected:
    subprocess.run([PYTHON,'-c',download_script,name,str(MEDIA)],env=hf_env,check=True)
print('Downloaded selected files:', len(selected))


## Extract and audit before training

The CUDA path can still be slow and memory intensive; verify the required GPU diagnostic and model smoke-test cells first. It must use the verified pretrained weights, and each failure halts the run; never relabel or silently discard failed records. Resolve extraction failures and re-audit provenance and split integrity before training. The JSON reports and features contain sensitive metadata; keep them private.


In [ ]:
DATASET_VERIFIED = False
assert globals().get('GPU_DIAGNOSTIC_PASSED', False) and globals().get('MODEL_SMOKE_PASSED', False), 'Run both GPU checks first'
env = dict(PROJECT_ENV, OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', XFORMERS_DISABLED='1', CUBLAS_WORKSPACE_CONFIG=':4096:8')
for r in records:
    out = Path(r['feature_path']); report = REPORTS / (r['sample_id'] + '.json')
    assert not out.exists() and not report.exists(), 'Choose fresh private outputs'
    with report.open('x') as stream:
        subprocess.run([PYTHON, '-m', 'deepfake_dtw.extract', r['video_path'], '--audio-checkpoint', str(WHISPER), '--visual-checkpoint', str(DINO), '--visual-format', 'dinov2_vits14', '--dinov2-source', str(DINO_SOURCE), '--visual-sha256', DINO_SHA256, '--authorized-models', '--device', 'cuda', '--output', str(out)], cwd=REPO, env=env, stdout=stream, check=True)
print('Feature extraction completed for all planned clips; inspect private quality reports.')


## Train and final evaluation

Audit labels, real/fake and manipulation counts, identities, original links, audio and provider split integrity before running. The CUDA trainer checks metadata and file hashes for all partitions and reads only training and validation feature arrays during fitting. Validation selects both epoch and threshold. Run final held-out evaluation only after all decisions are frozen. No score is calibrated by this workflow.


In [ ]:
# Validate all media/features and split provenance before training; inspect the private quality reports as well.
DATASET_VERIFIED = False
subprocess.run([PYTHON, '-m', 'deepfake_dtw.dataset', 'validate', str(PLAN)], cwd=REPO, env=env, check=True)
DATASET_VERIFIED = True


In [ ]:
assert globals().get('GPU_DIAGNOSTIC_PASSED', False) and globals().get('MODEL_SMOKE_PASSED', False) and globals().get('DATASET_VERIFIED', False), 'Complete GPU checks and dataset audit first'
checkpoint = PRIVATE / 'lavdf' / 'baseline.pt'
training_report = REPORTS / 'training.json'
assert not checkpoint.exists() and not training_report.exists()
with training_report.open('x') as stream:
    subprocess.run([PYTHON, '-m', 'deepfake_dtw.training', 'train', str(PLAN), str(checkpoint), '--seed', '42', '--epochs', '10', '--device', 'cuda'], cwd=REPO, env=env, stdout=stream, check=True)
print('Training report saved privately; review validation before final evaluation.')


In [ ]:
assert globals().get('DATASET_VERIFIED', False), 'Complete the dataset audit first'
# Run only after freezing all decisions. This writes sensitive failure IDs privately.
report = REPORTS / 'held-out-test.json'
assert checkpoint.is_file() and not report.exists()
with report.open('x') as stream:
    subprocess.run([PYTHON, '-m', 'deepfake_dtw.training', 'evaluate', str(PLAN), str(checkpoint), '--partition', 'test', '--device', 'cuda'], cwd=REPO, env=env, stdout=stream, check=True)
print('Held-out report saved privately.')
